# Chapter 4: Language Modeling

Language modeling is a fundamental task in NLP that involves predicting the next word or sequence of words in a sentence. It serves as the backbone for many advanced NLP applications such as speech recognition, machine translation, text generation, and more.

In this notebook, we will explore different techniques for building language models:

1. **N-grams** - Simple statistical models based on word sequences
2. **Hidden Markov Models (HMMs)** - Probabilistic models for sequence analysis
3. **Recurrent Neural Networks (RNNs)** - Neural networks designed for sequential data
4. **Long Short-Term Memory Networks (LSTMs)** - Advanced RNNs that handle long-term dependencies

## Setup

Install and import the required packages.

In [ ]:
!pip install nltk hmmlearn tensorflow numpy

In [ ]:
import numpy as np
from collections import defaultdict, Counter
import nltk
from nltk import ngrams
from nltk.util import ngrams as nltk_ngrams

nltk.download('punkt')
nltk.download('punkt_tab')

---
## 4.1 N-grams

N-grams are contiguous sequences of N items derived from a given sample of text. In NLP, these items are typically words.

- **Unigram (1-gram):** `"Natural"`
- **Bigram (2-gram):** `"Natural Language"`
- **Trigram (3-gram):** `"Natural Language Processing"`

N-grams capture local word dependencies by considering a fixed window of N words. They are widely used in text prediction, speech recognition, machine translation, and text generation.

### 4.1.1 Generating N-grams in Python

We use the `ngrams` function from NLTK to generate unigrams, bigrams, and trigrams from a sample text.

In [ ]:
# Sample text
text = "Natural Language Processing is a fascinating field of study."

# Tokenize the text into words
tokens = nltk.word_tokenize(text)

# Function to generate N-grams
def generate_ngrams(tokens, n):
    n_grams = ngrams(tokens, n)
    return [' '.join(grams) for grams in n_grams]

# Generate unigrams, bigrams, and trigrams
unigrams = generate_ngrams(tokens, 1)
bigrams = generate_ngrams(tokens, 2)
trigrams = generate_ngrams(tokens, 3)

print("Unigrams:")
print(unigrams)
print("\nBigrams:")
print(bigrams)
print("\nTrigrams:")
print(trigrams)

### 4.1.2 N-gram Language Models

N-gram language models predict the next word in a sequence based on the previous N-1 words. A **bigram model** (N=2) estimates:

$$P(w_n | w_{n-1}) = \frac{\text{count}(w_{n-1}, w_n)}{\text{count}(w_{n-1})}$$

For example, if the bigram "language processing" appears 50 times and "language" appears 200 times, then P(processing | language) = 50/200 = 0.25.

### 4.1.3 Training a Bigram Language Model

Training involves:
1. Tokenizing the corpus
2. Generating N-grams
3. Counting N-gram occurrences
4. Normalizing counts to obtain probabilities

In [ ]:
# Sample text corpus
corpus = [
    "Natural Language Processing is a fascinating field of study.",
    "Machine learning and NLP are closely related.",
    "Language models are essential for NLP tasks."
]

# Tokenize the text into words
tokenized_corpus = [nltk.word_tokenize(sentence) for sentence in corpus]
print("Tokenized corpus:")
for sent in tokenized_corpus:
    print(sent)

In [ ]:
# Function to calculate bigram probabilities
def train_bigram_model(tokenized_corpus):
    model = defaultdict(lambda: defaultdict(lambda: 0))

    # Count bigrams
    for sentence in tokenized_corpus:
        for w1, w2 in ngrams(sentence, 2):
            model[w1][w2] += 1

    # Calculate probabilities
    for w1 in model:
        total_count = float(sum(model[w1].values()))
        for w2 in model[w1]:
            model[w1][w2] /= total_count

    return model

# Train the bigram model
bigram_model = train_bigram_model(tokenized_corpus)

# Function to get the probability of a bigram
def get_bigram_probability(bigram_model, w1, w2):
    return bigram_model[w1][w2]

# Query some probabilities
print("Bigram Probability P(NLP | for):", get_bigram_probability(bigram_model, 'for', 'NLP'))
print("Bigram Probability P(Processing | Language):", get_bigram_probability(bigram_model, 'Language', 'Processing'))

In [ ]:
# Inspect the full bigram model
print("Bigram probability table:")
print("-" * 50)
for w1 in sorted(bigram_model.keys()):
    for w2 in sorted(bigram_model[w1].keys()):
        prob = bigram_model[w1][w2]
        if prob > 0:
            print(f"  P({w2:15s} | {w1:15s}) = {prob:.4f}")

### 4.1.4 Generalized N-gram Training

We can generalize the counting approach to work with any N.

In [ ]:
def count_ngrams(tokenized_corpus, n):
    """Count N-gram occurrences, keyed by context (first N-1 words)."""
    counts = defaultdict(lambda: defaultdict(int))
    for sentence in tokenized_corpus:
        for ngram in ngrams(sentence, n):
            counts[ngram[:-1]][ngram[-1]] += 1
    return counts

def calculate_probabilities(counts):
    """Normalize counts to probabilities."""
    probabilities = defaultdict(dict)
    for context in counts:
        total_count = float(sum(counts[context].values()))
        for word in counts[context]:
            probabilities[context][word] = counts[context][word] / total_count
    return probabilities

# Count and compute trigram probabilities
trigram_counts = count_ngrams(tokenized_corpus, 3)
trigram_probabilities = calculate_probabilities(trigram_counts)

# Example: probability of "Processing" given context ("Natural", "Language")
context = ('Natural', 'Language')
print(f"P(Processing | {context}) =", trigram_probabilities[context].get('Processing', 0))

### 4.1.5 Limitations of N-gram Models

While N-gram models are simple and effective, they have key limitations:

| Limitation | Description |
|---|---|
| **Sparsity** | As N increases, many N-grams never appear in training data |
| **Context limitation** | Only captures N-1 words of context |
| **Memory usage** | High-order models require significant storage |
| **No generalization** | Cannot handle unseen word sequences well |
| **No semantics** | Treats words as independent tokens without meaning |

These limitations motivate more advanced approaches like HMMs and neural language models.

---
## 4.2 Hidden Markov Models (HMMs)

Hidden Markov Models are statistical models used for sequence analysis. They model sequences of observable events (e.g., words) and the underlying hidden states (e.g., parts of speech) that generate them.

An HMM consists of:
- **States**: Hidden variables (e.g., Noun, Verb)
- **Observations**: Observable events (e.g., words)
- **Transition probabilities**: Likelihood of moving between hidden states
- **Emission probabilities**: Likelihood of an observation given a hidden state
- **Initial probabilities**: Likelihood of starting in each state

### 4.2.1 The Three Fundamental Problems of HMMs

1. **Evaluation Problem**: What is the probability of an observation sequence given the model? Solved with the **Forward algorithm**.
2. **Decoding Problem**: What is the most likely sequence of hidden states? Solved with the **Viterbi algorithm**.
3. **Learning Problem**: How to estimate model parameters from data? Solved with the **Baum-Welch (EM) algorithm**.

### 4.2.2 Implementing HMMs for POS Tagging

We use the `hmmlearn` library to build an HMM for part-of-speech tagging.

In [ ]:
from hmmlearn import hmm

# Define the states and observations
states = ["Noun", "Verb"]
n_states = len(states)

observations = ["I", "run", "to", "the", "store"]
n_observations = len(observations)

# Transition probability matrix (A)
# Rows = from state, Columns = to state
transition_probability = np.array([
    [0.7, 0.3],  # From Noun to [Noun, Verb]
    [0.4, 0.6]   # From Verb to [Noun, Verb]
])

# Emission probability matrix (B)
# Rows = state, Columns = observation
emission_probability = np.array([
    [0.2, 0.3, 0.2, 0.1, 0.2],  # Noun -> ["I", "run", "to", "the", "store"]
    [0.1, 0.6, 0.1, 0.1, 0.1]   # Verb -> ["I", "run", "to", "the", "store"]
])

# Initial probability vector (pi)
start_probability = np.array([0.6, 0.4])  # [Noun, Verb]

print("States:", states)
print("Observations:", observations)
print("\nTransition matrix:\n", transition_probability)
print("\nEmission matrix:\n", emission_probability)
print("\nStart probabilities:", start_probability)

In [ ]:
# Create the HMM model
model = hmm.CategoricalHMM(n_components=n_states)
model.startprob_ = start_probability
model.transmat_ = transition_probability
model.emissionprob_ = emission_probability

# Encode the observations to integers
observation_sequence = np.array([0, 1, 2, 3, 4]).reshape(-1, 1)  # "I", "run", "to", "the", "store"

# Predict the hidden states using the Viterbi algorithm (Decoding Problem)
logprob, hidden_states = model.decode(observation_sequence, algorithm="viterbi")

print("Observations:", [observations[i] for i in observation_sequence.flatten()])
print("Hidden states:", [states[i] for i in hidden_states])
print(f"Log probability: {logprob:.4f}")

### 4.2.3 Learning HMM Parameters from Data

Instead of manually specifying the transition and emission matrices, we can learn them from training data using the Baum-Welch algorithm.

In [ ]:
# Sample data: multiple sequences of observations
training_sequences = [
    [0, 1, 2, 3, 4],  # "I run to the store"
    [4, 2, 0, 1, 3],  # "store to I run the"
    [1, 2, 3, 0, 4],  # "run to the I store"
]

# Convert to format suitable for hmmlearn
training_sequences = [np.array(seq).reshape(-1, 1) for seq in training_sequences]
lengths = [len(seq) for seq in training_sequences]
training_data = np.concatenate(training_sequences)

# Create and train the HMM model
learned_model = hmm.CategoricalHMM(n_components=n_states, n_iter=100, random_state=42)
learned_model.fit(training_data, lengths)

print("Learned start probabilities:")
print(learned_model.startprob_.round(4))
print("\nLearned transition probabilities:")
print(learned_model.transmat_.round(4))
print("\nLearned emission probabilities:")
print(learned_model.emissionprob_.round(4))

---
## 4.3 Recurrent Neural Networks (RNNs)

RNNs are neural networks designed for processing sequential data. Unlike feedforward networks, RNNs have connections that form directed cycles, enabling them to maintain a **hidden state** that captures information about previous inputs.

At each time step *t*, the hidden state is updated:

$$h_t = \sigma(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$

This makes RNNs well-suited for tasks involving time series, text, speech, and other sequential data.

### 4.3.1 Challenges with RNNs

- **Vanishing gradients**: Gradients shrink during backpropagation through time, making it hard to learn long-range dependencies
- **Exploding gradients**: Gradients grow exponentially, causing unstable training
- **Computational cost**: Sequential processing prevents parallelization

These challenges motivate advanced architectures like LSTMs and GRUs.

### 4.3.2 Implementing an RNN for Text Generation

We build a character-level RNN using TensorFlow/Keras that learns to predict the next character in a sequence.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Sample text corpus
text = "hello world"

# Create a character-level vocabulary
chars = sorted(set(text))
char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

print(f"Vocabulary ({len(chars)} chars): {chars}")
print(f"char_to_idx: {char_to_idx}")

In [ ]:
# Create input-output pairs for training
sequence_length = 3
X = []
y = []

for i in range(len(text) - sequence_length):
    input_seq = text[i:i + sequence_length]
    target_char = text[i + sequence_length]
    X.append([char_to_idx[char] for char in input_seq])
    y.append(char_to_idx[target_char])
    print(f"  '{input_seq}' -> '{target_char}'")

X = np.array(X)
y = to_categorical(y, num_classes=len(chars))

# Reshape input to be compatible with RNN: (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Define the RNN model
rnn_model = Sequential([
    SimpleRNN(50, input_shape=(sequence_length, 1)),
    Dense(len(chars), activation='softmax')
])

rnn_model.compile(optimizer='adam', loss='categorical_crossentropy')
rnn_model.summary()

In [ ]:
# Train the model
history = rnn_model.fit(X, y, epochs=200, verbose=0)

print(f"Final loss: {history.history['loss'][-1]:.4f}")

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'])
plt.title('RNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Function to generate text using the trained model
def generate_text(model, start_string, num_generate, char_to_idx, idx_to_char):
    input_eval = [char_to_idx[s] for s in start_string]
    input_eval = np.array(input_eval).reshape((1, len(input_eval), 1))
    text_generated = []

    for i in range(num_generate):
        predictions = model.predict(input_eval, verbose=0)
        predicted_id = np.argmax(predictions[-1])
        input_eval = np.append(input_eval[:, 1:], [[predicted_id]], axis=1)
        text_generated.append(idx_to_char[predicted_id])

    return start_string + ''.join(text_generated)

# Generate new text
start_string = "hel"
generated_text = generate_text(rnn_model, start_string, 8, char_to_idx, idx_to_char)
print("Generated text:", repr(generated_text))

---
## 4.4 Long Short-Term Memory Networks (LSTMs)

LSTMs are a special type of RNN designed to address the vanishing gradient problem and capture long-term dependencies. They introduce **memory cells** controlled by three gates:

- **Forget gate** ($f_t$): Decides what information to discard from the cell state
- **Input gate** ($i_t$): Decides which new information to store in the cell state
- **Output gate** ($o_t$): Decides what to output based on the cell state

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$

### 4.4.1 Implementing an LSTM for Text Generation

The LSTM implementation is very similar to the RNN, but we replace `SimpleRNN` with `LSTM`. This single change gives the model the gating mechanisms needed to handle longer dependencies.

In [ ]:
from tensorflow.keras.layers import LSTM

# We reuse the same text, vocabulary, and training data (X, y) from the RNN section.
# Rebuild to keep this section self-contained:

text = "hello world"
chars = sorted(set(text))
char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

sequence_length = 3
X_lstm = []
y_lstm = []
for i in range(len(text) - sequence_length):
    X_lstm.append([char_to_idx[char] for char in text[i:i + sequence_length]])
    y_lstm.append(char_to_idx[text[i + sequence_length]])

X_lstm = np.array(X_lstm)
y_lstm = to_categorical(y_lstm, num_classes=len(chars))
X_lstm = X_lstm.reshape((X_lstm.shape[0], X_lstm.shape[1], 1))

print(f"Training data: {X_lstm.shape[0]} samples, sequence length {sequence_length}")

In [ ]:
# Define the LSTM model
lstm_model = Sequential([
    LSTM(50, input_shape=(sequence_length, 1)),
    Dense(len(chars), activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='categorical_crossentropy')
lstm_model.summary()

In [ ]:
# Train the LSTM model
lstm_history = lstm_model.fit(X_lstm, y_lstm, epochs=200, verbose=0)

print(f"Final loss: {lstm_history.history['loss'][-1]:.4f}")

In [ ]:
# Compare RNN vs LSTM training curves
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='SimpleRNN', alpha=0.8)
plt.plot(lstm_history.history['loss'], label='LSTM', alpha=0.8)
plt.title('Training Loss: RNN vs LSTM')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Generate text with the LSTM model
start_string = "hel"
generated_lstm = generate_text(lstm_model, start_string, 8, char_to_idx, idx_to_char)
print("LSTM generated text:", repr(generated_lstm))

### 4.4.2 Evaluating LSTM Performance

Key evaluation approaches:

| Method | Purpose |
|---|---|
| **Loss** | Track categorical cross-entropy during training |
| **Accuracy** | Proportion of correct next-character predictions |
| **Perplexity** | $2^{H(p)}$ -- lower is better; measures how "surprised" the model is |
| **Qualitative** | Inspect generated text for coherence |
| **Train/Val curves** | Monitor for overfitting |

Regularization techniques (dropout, early stopping) help prevent overfitting.

### 4.4.3 Applications of LSTMs

LSTMs are used across many domains:

- **Text generation** -- creative writing, chatbots, automated content
- **Machine translation** -- sequence-to-sequence models (e.g., Google Translate)
- **Speech recognition** -- converting audio to text (Siri, Alexa)
- **Time series forecasting** -- stock prices, weather, health metrics
- **Sentiment analysis** -- classifying text polarity
- **Music generation** -- composing melodies from learned patterns
- **Anomaly detection** -- fraud detection, network security

---
## Practical Exercises

### Exercise 1: Generate Trigrams

**Task:** Generate trigrams from the text `"Natural Language Processing with Python"`.

In [ ]:
# Exercise 1 Solution
text_ex1 = "Natural Language Processing with Python"
tokens_ex1 = nltk.word_tokenize(text_ex1)

trigrams_ex1 = ngrams(tokens_ex1, 3)

print("Trigrams:")
for gram in trigrams_ex1:
    print(gram)

### Exercise 2: Bigram Language Model

**Task:** Train a bigram model on the corpus below and calculate P(Processing | Language).

```python
corpus = [
    "Natural Language Processing is fascinating.",
    "Language models are important in NLP.",
    "Machine learning and NLP are closely related."
]
```

In [ ]:
# Exercise 2 Solution
corpus_ex2 = [
    "Natural Language Processing is fascinating.",
    "Language models are important in NLP.",
    "Machine learning and NLP are closely related."
]

tokenized_ex2 = [nltk.word_tokenize(sentence) for sentence in corpus_ex2]

bigram_model_ex2 = train_bigram_model(tokenized_ex2)

print("Bigram Probability P(Processing | Language):")
print(get_bigram_probability(bigram_model_ex2, 'Language', 'Processing'))

### Exercise 3: HMM for Part-of-Speech Tagging

**Task:** Train an HMM on two tagged sentences and predict the POS tags.

```
Sentences: ["I", "run", "to", "the", "store"], ["She", "jumps", "over", "the", "fence"]
Tags:      [PRON, VERB, ADP, DET, NOUN],       [PRON, VERB, ADP, DET, NOUN]
```

In [ ]:
# Exercise 3 Solution
states_ex3 = ["PRON", "VERB", "ADP", "DET", "NOUN"]
n_states_ex3 = len(states_ex3)

observations_ex3 = ["I", "run", "to", "the", "store", "She", "jumps", "over", "fence"]
observation_to_idx = {obs: idx for idx, obs in enumerate(observations_ex3)}

sentences_ex3 = [
    ["I", "run", "to", "the", "store"],
    ["She", "jumps", "over", "the", "fence"]
]

# Encode observations
X_ex3 = [[observation_to_idx[word] for word in sent] for sent in sentences_ex3]
X_ex3_concat = np.concatenate([np.array(x).reshape(-1, 1) for x in X_ex3])
lengths_ex3 = [len(x) for x in sentences_ex3]

# Train HMM
model_ex3 = hmm.CategoricalHMM(n_components=n_states_ex3, n_iter=100, random_state=42)
model_ex3.fit(X_ex3_concat, lengths_ex3)

# Predict hidden states
logprob_ex3, hidden_states_ex3 = model_ex3.decode(X_ex3_concat, algorithm="viterbi")
predicted_tags = [states_ex3[s] for s in hidden_states_ex3]

print("Observations:", sentences_ex3[0] + sentences_ex3[1])
print("Predicted tags:", predicted_tags)

### Exercise 4: RNN for Text Generation

**Task:** Train a SimpleRNN on `"hello world"` and generate text starting from `"hel"`.

In [ ]:
# Exercise 4 Solution
# (This reuses the RNN model trained above. If starting fresh, re-run the RNN section.)
generated_rnn = generate_text(rnn_model, "hel", 5, char_to_idx, idx_to_char)
print("RNN generated text:", repr(generated_rnn))

### Exercise 5: LSTM for Text Generation

**Task:** Train an LSTM on `"hello world"` and generate text starting from `"hel"`.

In [ ]:
# Exercise 5 Solution
# (This reuses the LSTM model trained above. If starting fresh, re-run the LSTM section.)
generated_lstm_ex5 = generate_text(lstm_model, "hel", 5, char_to_idx, idx_to_char)
print("LSTM generated text:", repr(generated_lstm_ex5))

---
## Summary

| Model | Strengths | Weaknesses |
|---|---|---|
| **N-grams** | Simple, fast, interpretable | Sparsity, no long-range context, no semantics |
| **HMMs** | Probabilistic, handles hidden states | Assumes Markov property, limited expressiveness |
| **RNNs** | Handles variable-length sequences | Vanishing/exploding gradients |
| **LSTMs** | Captures long-term dependencies | More parameters, slower to train |

Each technique builds on the limitations of the previous one. N-grams provide a statistical baseline, HMMs add hidden state modeling, RNNs introduce neural sequence processing, and LSTMs solve the gradient problems that limit standard RNNs.